In [17]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 47), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 10.24 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrank-ml-internship/flyrank-ml-internship


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row = one content page (content_hash_id) for one
client (client_hash_id), for one day (report_date) the daily fact
table's natural grain.

Time window: month = 2026-03 (a mid-panel month, chosen deliberately
instead of the final sample month, so I don't accidentally train or
verify on the same window I'd later treat as a future outcome).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature fields (known before the decision point):**
- impressions, clicks, average position, CTR
- sessions, engagement_rate, scroll_rate
- content_age_days, word_count, freshness tier

**Label/proxy field:**
- trend_direction (used to build is_declining_label) — this is what
  I'm trying to predict/rank pages by

**Context fields (background, not used as model input):**
- content_type, main_intent, competition_level
- these help explain results but aren't fed into the model directly

**Excluded fields:**
- Any FlyRank product decision output (health_score, priority_score,
  action_type, refresh flags). These are excluded because they are the
  product's own conclusion, not raw evidence — using them as a feature
  would let the model just copy an existing answer instead of learning
  from real signals.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB + HF token ready")

DuckDB + HF token ready


In [21]:
schema_check = con.execute("""
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/**/*fact_content_daily_performance*.parquet')
LIMIT 5
""").df()

print(schema_check.columns.tolist())
print(schema_check)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
  report_date           client_hash_id           content_hash_id  \
0  2026-06-01  client_3ffa76342f366962  content_1a6296faee432dae   
1  2026-06-01  client_3ffa76342f366962  content_73f21e612565035a   
2  2026-06-01  client_3ffa76342f366962  content_5a5be514ff559598   
3  2026-06-01  client_3ffa76342f366962  content_05b377d0c8a5cfd8   
4  2026-06-01  client_3ffa76342f366962  content_dc34c661d63e55a9   

   client_has_gsc  client_has_ga4  gsc_data_available  g

In [22]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

for f in files['file']:
    print(f)

hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_co

In [23]:
query1 = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) as row_count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 10
"""

result1 = con.execute(query1).df()
print("Duplicate grain check (should be empty if grain is clean):")
print(result1)
print("\nNumber of duplicate combos found:", len(result1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain check (should be empty if grain is clean):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

Number of duplicate combos found: 0


In [24]:
query2 = """
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date,
    COUNT(DISTINCT client_hash_id) as num_clients,
    COUNT(DISTINCT content_hash_id) as num_content_items
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
"""

result2 = con.execute(query2).df()
print("Row count and date span for month=2026-03:")
print(result2)

Row count and date span for month=2026-03:
   total_rows earliest_date latest_date  num_clients  num_content_items
0     9841378    2026-03-01  2026-03-31           55             331437


In [25]:
query3 = """
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_gsc,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
"""

result3 = con.execute(query3).df()
print("Availability check for month=2026-03:")
print(result3)
print("\nGSC availability %:", round(result3['rows_with_gsc'][0] / result3['total_rows'][0] * 100, 1))
print("GA4 availability %:", round(result3['rows_with_ga4'][0] / result3['total_rows'][0] * 100, 1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability check for month=2026-03:
   total_rows  rows_with_gsc  rows_with_ga4
0     9841378      3611061.0       413966.0

GSC availability %: 36.7
GA4 availability %: 4.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Five Features (built from month=2026-03 data)**

1. gsc_impressions — known before decision point? Yes, it's an
   observed measurement from the feature window itself.
2. gsc_clicks — known before decision point? Yes, same as above.
3. gsc_avg_position — known before decision point? Yes, an observed
   search ranking metric from the feature window.
4. content_age_days (derived from content creation date) — known
   before decision point? Yes, purely a function of time, always
   available.
5. gsc_ctr (clicks/impressions) — known before decision point? Yes,
   derived only from the same-window observed signals above.

In [26]:
features_query = """
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    CASE WHEN gsc_impressions > 0 THEN gsc_sum_position / gsc_impressions ELSE NULL END as gsc_avg_position,
    CASE WHEN gsc_impressions > 0 THEN gsc_clicks / gsc_impressions ELSE NULL END as gsc_ctr
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
WHERE gsc_data_available IS TRUE
LIMIT 1000
"""

features_df = con.execute(features_query).df()
print("Feature frame shape:", features_df.shape)
features_df.head()

Feature frame shape: (1000, 7)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,gsc_ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,3.350000,0.000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,0.000000,0.000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,4.928000,0.008
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,4.000000,0.000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,2.272727,0.000


# **The Leakage Trap**

To demonstrate why leakage is dangerous, I will deliberately add one
label-derived column to my feature set: a column built directly from
trend_direction (the same field my label comes from). I expect this
to make a quick score look almost perfect — which is the warning sign
of leakage, not a real win. I will then remove it and report the
honest score without it.

In [27]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Build a quick label using trend info from the daily table
label_query = """
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    CASE WHEN gsc_impressions > 0 THEN gsc_sum_position / gsc_impressions ELSE NULL END as gsc_avg_position,
    CASE WHEN gsc_impressions > 0 THEN gsc_clicks / gsc_impressions ELSE NULL END as gsc_ctr,
    CASE WHEN gsc_clicks < gsc_impressions * 0.01 THEN 1 ELSE 0 END as is_declining_label
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
LIMIT 5000
"""

trap_df = con.execute(label_query).df().dropna()

# TRUE honest features - excluding the exact columns the label is computed from
X_honest = trap_df[['gsc_sum_position', 'gsc_avg_position']]
y = trap_df['is_declining_label']

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print("HONEST score (no leakage):", round(honest_score, 3))

HONEST score (no leakage): 0.564


In [28]:
# CHEATING VERSION: adding gsc_clicks back in — this is derived from
# the exact same signal the label is built on
X_cheat = trap_df[['gsc_sum_position', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_cheat, y, test_size=0.3, random_state=42)
model2 = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
cheat_score = roc_auc_score(y_test2, model2.predict_proba(X_test2)[:,1])
print("CHEATING score (with leakage):", round(cheat_score, 3))

print("\n--- Comparison ---")
print("Honest score:", round(honest_score, 3))
print("Cheating score:", round(cheat_score, 3))
print("This jump shows leakage: the label is literally computed from")
print("gsc_clicks and gsc_impressions, so including them as features")
print("lets the model 'cheat' by reading the answer directly.")

CHEATING score (with leakage): 1.0

--- Comparison ---
Honest score: 0.564
Cheating score: 1.0
This jump shows leakage: the label is literally computed from
gsc_clicks and gsc_impressions, so including them as features
lets the model 'cheat' by reading the answer directly.


# **Leakage Trap — Result**

Honest score (position-based features only): 0.564
Cheating score (with gsc_clicks and gsc_impressions added): 1.0

The jump from 0.564 to 1.0 is the warning sign of leakage, not a real
improvement. It happened because the label itself
(is_declining_label) is directly computed from gsc_clicks and
gsc_impressions — so feeding those same columns back in as features
lets the model simply read the answer instead of learning a pattern.

Decision: I remove gsc_clicks and gsc_impressions from my feature set
going forward and keep the honest score of 0.564 as the real baseline
for this label. This is the leakage check every future model in this
project must pass before I trust its numbers.

# **Data limits for this slice (month=2026-03):**
- GA4 (sessions, engagement, scroll) data is only available in 4.2%
  of rows this month — far sparser than search console data (36.7%).
  Any feature using sessions or engagement will only apply to a small
  fraction of pages, so a model relying heavily on those signals would
  have limited coverage.

- This is an unbalanced panel: 55 clients appear this month, but each
  client's tracking history started at a different time, so early rows
  for newer clients may show gsc_data_available = FALSE even though
  the page existed — this is "no tracking yet," not "no traffic."

- This single month (March 2026) cannot show seasonality or long-term
  trend — only 9 of 70 total clients have 12+ months of history, so
  seasonal patterns would need a wider date range than this slice.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.